# MSTR 用户批量小写化执行 Notebook

本 Notebook 用于在 MicroStrategy / Strategy One 环境中，逐步筛选包含大写字母的用户，并由执行人输入编号确认后，实际将选中的用户字段小写化。

执行原则：先看清单，再选编号，最后实际执行。不要跳 cell 执行。

## 1. 导入依赖并定义工具函数

如果这里报 `ModuleNotFoundError`，请先在当前 Python 环境执行：

```bash
pip install -r requirements.txt
```

In [ ]:
import os
import re
from dataclasses import dataclass
from getpass import getpass
from typing import Iterable

import pandas as pd
from IPython.display import display
from mstrio.connection import Connection
from mstrio.users_and_groups.user import User, list_users

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

SYSTEM_USER_NAMES = {
    "administrator",
    "guest",
    "public / guest",
    "public/guest",
    "system administrator",
    "mstr user",
    "microstrategy administrator",
}

UPPERCASE_RE = re.compile(r"[A-Z]")

@dataclass(frozen=True)
class UserChange:
    id: str
    display_name: str
    login_id: str
    trust_id: str
    new_display_name: str
    new_login_id: str
    new_trust_id: str


def has_uppercase(value: object) -> bool:
    return isinstance(value, str) and bool(UPPERCASE_RE.search(value))


def lowercase_or_none(value: object) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    return str(value).lower()


def users_to_dataframe(users: list[User]) -> pd.DataFrame:
    rows = []
    for user in users:
        rows.append(
            {
                "id": getattr(user, "id", None),
                "display_name": getattr(user, "name", None) or getattr(user, "full_name", None),
                "full_name": getattr(user, "full_name", None),
                "login_id": getattr(user, "username", None) or getattr(user, "abbreviation", None),
                "account_login": getattr(user, "abbreviation", None),
                "trust_id": getattr(user, "trust_id", None),
                "enabled": getattr(user, "enabled", None),
            }
        )
    return pd.DataFrame(rows)


def find_uppercase_users(df: pd.DataFrame) -> pd.DataFrame:
    text_columns = ["display_name", "full_name", "login_id", "account_login", "trust_id"]
    selected = df[text_columns]
    if hasattr(selected, "map"):
        uppercase_cells = selected.map(has_uppercase)
    else:
        uppercase_cells = selected.applymap(has_uppercase)
    return df[uppercase_cells.any(axis=1)].copy()


def filter_system_users(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    def is_system(row: pd.Series) -> bool:
        values = {
            str(row.get("display_name") or "").strip().lower(),
            str(row.get("full_name") or "").strip().lower(),
            str(row.get("login_id") or "").strip().lower(),
            str(row.get("account_login") or "").strip().lower(),
        }
        return bool(values & SYSTEM_USER_NAMES)

    system_mask = df.apply(is_system, axis=1)
    return df[~system_mask].copy(), df[system_mask].copy()


def build_changes(df: pd.DataFrame) -> list[UserChange]:
    changes = []
    for row in df.itertuples(index=False):
        display_name = row.display_name or row.full_name or row.login_id
        login_id = row.login_id or row.account_login
        trust_id = row.trust_id
        changes.append(
            UserChange(
                id=row.id,
                display_name=display_name or "",
                login_id=login_id or "",
                trust_id=trust_id or "",
                new_display_name=lowercase_or_none(display_name) or "",
                new_login_id=lowercase_or_none(login_id) or "",
                new_trust_id=lowercase_or_none(trust_id) or "",
            )
        )
    return changes


def validate_no_login_collisions(all_users_df: pd.DataFrame, changes: Iterable[UserChange]) -> None:
    changes = list(changes)
    changed_ids = {change.id for change in changes}
    existing_logins = {
        str(row.login_id).lower(): row.id
        for row in all_users_df.itertuples(index=False)
        if row.id not in changed_ids and pd.notna(row.login_id) and row.login_id
    }

    seen_in_batch: dict[str, str] = {}
    errors = []
    for change in changes:
        login = change.new_login_id
        if not login:
            errors.append(f"{change.display_name}: 登录 ID 为空，无法更新")
            continue
        if login in existing_logins:
            errors.append(f"{change.login_id} -> {login}: 与现有用户 ID {existing_logins[login]} 冲突")
        if login in seen_in_batch:
            errors.append(f"{change.login_id} -> {login}: 与本批次用户 ID {seen_in_batch[login]} 冲突")
        seen_in_batch[login] = change.id

    if errors:
        message = "发现小写化后的登录 ID 冲突，已停止执行：\n" + "\n".join(f"- {error}" for error in errors)
        raise RuntimeError(message)


def parse_selection(raw_value: str, valid_numbers: set[int]) -> list[int]:
    value = raw_value.strip().lower()
    if value in {"all", "全部"}:
        return sorted(valid_numbers)
    if not value:
        raise ValueError("输入不能为空。示例：1 或 1;3;5 或 all")

    parts = [part.strip() for part in value.split(";")]
    if any(not part for part in parts):
        raise ValueError("编号之间请用英文分号分隔，且不要留空。示例：1;3;5")
    if any(not part.isdigit() for part in parts):
        raise ValueError("只能输入数字编号、英文分号，或者 all。示例：1 或 1;3;5 或 all")

    selected = [int(part) for part in parts]
    invalid = [number for number in selected if number not in valid_numbers]
    if invalid:
        raise ValueError(f"以下编号不存在：{invalid}。请从表格中的 batch_no 选择。")

    return sorted(set(selected))


def ask_user_selection(valid_numbers: set[int]) -> list[int]:
    sample = "输入示例：单个编号 1；多个编号 1;3;5；全部 all"
    while True:
        raw_value = input(f"请输入要实际小写化的用户编号，或输入 all 选择全部。{sample}\n你的输入: ")
        try:
            return parse_selection(raw_value, valid_numbers)
        except ValueError as exc:
            print(f"输入无效：{exc}")
            print(sample)


def apply_changes(connection: Connection, changes: list[UserChange]) -> pd.DataFrame:
    results = []
    for change in changes:
        print(f"正在处理: {change.display_name} / {change.login_id} -> {change.new_login_id}")
        try:
            user = User(connection=connection, id=change.id)
            user.alter(
                username=change.new_login_id,
                full_name=change.new_display_name,
                trust_id=change.new_trust_id or None,
                journal_comment="Batch lowercase user display name, login ID and trusted auth ID.",
            )
            results.append(
                {
                    "display_name": change.display_name,
                    "login_id": change.login_id,
                    "new_display_name": change.new_display_name,
                    "new_login_id": change.new_login_id,
                    "new_trust_id": change.new_trust_id,
                    "success": True,
                    "message": "updated",
                }
            )
        except Exception as exc:
            results.append(
                {
                    "display_name": change.display_name,
                    "login_id": change.login_id,
                    "new_display_name": change.new_display_name,
                    "new_login_id": change.new_login_id,
                    "new_trust_id": change.new_trust_id,
                    "success": False,
                    "message": str(exc),
                }
            )
    return pd.DataFrame(results)

print(f"pandas version: {pd.__version__}")
print("工具函数加载完成。")

## 2. 配置并登录 MSTR Library

密码不会写入 Notebook 文件。建议由执行人在运行 cell 时输入密码，或提前设置环境变量 `MSTR_PASSWORD`。

In [ ]:
library_url = os.getenv("MSTR_LIBRARY_URL", "https://env-363414.customer.cloud.microstrategy.com/MicroStrategyLibrary/")
username = os.getenv("MSTR_USERNAME", "mstr")
password = os.getenv("MSTR_PASSWORD") or getpass("请输入 MSTR 登录密码: ")
login_mode = int(os.getenv("MSTR_LOGIN_MODE", "1"))

connection = Connection(
    base_url=library_url,
    username=username,
    password=password,
    login_mode=login_mode,
)

print(f"已连接: {library_url}")

## 3. 列出当前平台全部用户并导入 DataFrame

确认这里能看到用户清单后，再继续下一步。

In [ ]:
users = list_users(connection=connection)
users_df = users_to_dataframe(users)

print(f"当前平台用户数量: {len(users_df)}")
display(users_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

## 4. 筛选包含大写字母的用户

筛选范围包括显示名、登录 ID、account login、受信任验证请求用户 ID。

In [ ]:
uppercase_df = find_uppercase_users(users_df)

print(f"包含大写字母的用户数量: {len(uppercase_df)}")
display(uppercase_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

## 5. 选择本次要实际小写化的用户

本步骤会：

1. 默认剔除疑似 MSTR 平台自带用户。
2. 给每个候选用户生成 `batch_no` 编号。
3. 要求执行人输入要实际处理的编号。

输入示例：

- `1`：只处理编号 1。
- `1;3;5`：处理编号 1、3、5。
- `all`：处理全部候选用户。

如果输入错误，cell 会提示错误并要求重新输入。

In [ ]:
exclude_system_users = True

candidate_df, system_df = filter_system_users(uppercase_df)

print(f"疑似平台自带用户数量: {len(system_df)}")
if not system_df.empty:
    display(system_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

selectable_df = candidate_df.copy() if exclude_system_users else uppercase_df.copy()
selectable_df = selectable_df.reset_index(drop=True)
selectable_df.insert(0, "batch_no", range(1, len(selectable_df) + 1))

print(f"可选择的小写化候选用户数量: {len(selectable_df)}")
display(selectable_df[["batch_no", "display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

if selectable_df.empty:
    raise RuntimeError("没有可处理用户。请停止执行后续 cell。")

valid_numbers = set(selectable_df["batch_no"].tolist())
selected_numbers = ask_user_selection(valid_numbers)

target_df = selectable_df[selectable_df["batch_no"].isin(selected_numbers)].drop(columns=["batch_no"]).copy()

print(f"本次确认要实际小写化的用户数量: {len(target_df)}")
display(selectable_df[selectable_df["batch_no"].isin(selected_numbers)][["batch_no", "display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

## 6. 生成最终变更清单并检查冲突

这一步不会修改 MSTR，只负责展示“旧值 -> 新值”并检查小写后的登录 ID 是否冲突。

In [ ]:
changes = build_changes(target_df)
validate_no_login_collisions(users_df, changes)

preview_df = pd.DataFrame([change.__dict__ for change in changes])

print(f"最终待执行用户数量: {len(preview_df)}")
display(preview_df[[
    "display_name",
    "login_id",
    "trust_id",
    "new_display_name",
    "new_login_id",
    "new_trust_id",
]].fillna(""))

## 7. 实际执行小写化

执行这个 cell 会真实修改 MSTR 用户。请只在第 6 步变更清单确认无误后执行。

In [ ]:
result_df = apply_changes(connection, changes)

print("执行结果：")
display(result_df.fillna(""))

if not result_df["success"].all():
    raise RuntimeError("存在执行失败项，请查看 result_df 的 message 字段。")

## 8. 再次列出当前平台用户并检查结果

如果本批次目标用户中仍包含大写字母的数量为 `0`，说明本批次处理成功。

In [ ]:
refreshed_users = list_users(connection=connection)
refreshed_df = users_to_dataframe(refreshed_users)

print(f"当前平台用户数量: {len(refreshed_df)}")
display(refreshed_df[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

remaining_targets = find_uppercase_users(refreshed_df)
processed_ids = {change.id for change in changes}
remaining_processed = remaining_targets[remaining_targets["id"].isin(processed_ids)]

print(f"本批次目标用户中仍包含大写字母的数量: {len(remaining_processed)}")
display(remaining_processed[["display_name", "login_id", "account_login", "trust_id", "enabled"]].fillna(""))

if remaining_processed.empty:
    print("检查成功：本批次目标用户已完成小写化。")
    print("恭喜，批处理脚本运行成功！")
else:
    raise RuntimeError("检查失败：部分本批次目标用户仍包含大写字母。")